# Module 2: Shrinking the Bytes — Attention Variants and Quantization

**ODSC 2026** | Serving LLMs Efficiently at Scale | [Harshul Jain](https://github.com/harshuljain13)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/harshuljain13/llm-inference-at-scale/blob/master/workshop/odsc-2026/demos/odsc_2026_module2_attention_and_quantization.ipynb)
[![Open In Molab](https://img.shields.io/badge/Open%20in-Molab-blue)](https://molab.marimo.io/github/harshuljain13/llm-inference-at-scale/blob/master/workshop/odsc-2026/demos/odsc_2026_module2_attention_and_quantization.ipynb)

Module 1 ended on a number that does not fit: **80 users at 16K context needs 160 GiB of
KV cache**, against a single GPU that has nowhere near that free once the weights are
loaded. Both terms of that product are attackable, and there are exactly two ways in.

| Lever | Shrinks | Mechanism | Measured in |
|-------|---------|-----------|-------------|
| **Attention variant** | the KV cache | fewer KV heads (GQA/MQA) or a compressed latent (MLA) | §2, §3 |
| **Quantization** | the weights | fewer bits per parameter (INT8, INT4) | §5 |

They are independent, they stack, and §6 puts both on the same axis — recomputing
A §06's capacity table with each lever applied.

**Same ground rules as Module 1:** measured numbers only. The one figure we cannot run
here (MLA, which needs DeepSeek weights) is labelled as derived from published config,
not presented as a benchmark.

## 0. Setup

In [ ]:
import subprocess, sys

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                       'transformers', 'torch', 'matplotlib', 'pandas', 'accelerate'])

import gc
import time

import pandas as pd
import torch
import matplotlib.pyplot as plt
from transformers import AutoConfig, AutoModelForCausalLM, AutoTokenizer

HAS_CUDA = torch.cuda.is_available()
DEVICE = torch.device('cuda' if HAS_CUDA else 'cpu')
DTYPE = torch.float16 if HAS_CUDA else torch.float32

if HAS_CUDA:
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'Device: {gpu_name} ({vram_gb:.1f} GB VRAM)')
else:
    gpu_name, vram_gb = 'CPU', 0.0
    print('Device: CPU. Sections 2-4 still run; Section 5 (quantization) needs CUDA.')

PROMPT_TOKENS = 256      # context length used for every KV measurement
DECODE_STEPS = 16
print(f'Measuring KV at {PROMPT_TOKENS} tokens of context, {DECODE_STEPS} decode steps.')

In [ ]:
# --- Plot helpers ---------------------------------------------------------
# Every chart in this notebook is one call into these. Keeps the story cells
# readable instead of burying the point under axis bookkeeping.
PALETTE = ['#dbeafe', '#dcfce7', '#fef3c7', '#ffedd5', '#ffe4e6', '#f3e8ff']
COOL, WARM = '#2563eb', '#991b1b'


def panels(n=1, width=6.5, height=4.5):
    fig, axes = plt.subplots(1, n, figsize=(width * n, height))
    return (axes,) if n == 1 else axes


def _finish(ax, title, xlabel=None, ylabel=None, legend=False):
    ax.set_title(title, fontsize=12, fontweight='bold')
    if xlabel:
        ax.set_xlabel(xlabel, fontsize=11)
    if ylabel:
        ax.set_ylabel(ylabel, fontsize=11)
    if legend:
        ax.legend(fontsize=9)
    for side in ('top', 'right'):
        ax.spines[side].set_visible(False)


def bars(ax, labels, values, title, ylabel, fmt='{:.0f}'):
    values = list(values)
    ax.bar(range(len(values)), values, color=PALETTE[:len(values)],
           edgecolor='#000', linewidth=1.2)
    ax.set_xticks(range(len(values)))
    ax.set_xticklabels(labels)
    for i, v in enumerate(values):
        ax.text(i, v, fmt.format(v), ha='center', va='bottom', fontsize=10)
    ax.set_ylim(0, max(values) * 1.25)
    _finish(ax, title, ylabel=ylabel)


def hbars(ax, labels, values, title, xlabel, notes=None, fmt='{:.2f}'):
    values = list(values)
    ax.barh(range(len(values)), values, color=PALETTE[:len(values)],
            edgecolor='#000', linewidth=1.2)
    ax.set_yticks(range(len(values)))
    ax.set_yticklabels(labels)
    ax.invert_yaxis()
    for i, v in enumerate(values):
        label = fmt.format(v) + (f'  ({notes[i]})' if notes else '')
        ax.text(v + max(values) * 0.02, i, label, va='center', fontsize=9)
    ax.set_xlim(0, max(values) * 1.4)
    _finish(ax, title, xlabel=xlabel)


def stacked(ax, labels, parts, title, ylabel, fmt=None):
    # parts: [(series_name, [values aligned to labels]), ...]
    bottom = [0.0] * len(labels)
    for i, (name, vals) in enumerate(parts):
        ax.bar(labels, vals, bottom=bottom, color=PALETTE[i % len(PALETTE)],
               edgecolor='#000', linewidth=1.2, label=name)
        bottom = [b + v for b, v in zip(bottom, vals)]
    if fmt:
        for i, total in enumerate(bottom):
            ax.text(i, total, fmt.format(total), ha='center', va='bottom', fontsize=9)
    _finish(ax, title, ylabel=ylabel, legend=True)


def grouped(ax, groups, series, title, ylabel, fmt='{:.1f}'):
    # series: [(series_name, [values aligned to groups]), ...]
    width = 0.8 / len(series)
    for si, (name, vals) in enumerate(series):
        xs = [g + si * width - 0.4 + width / 2 for g in range(len(groups))]
        ax.bar(xs, vals, width, color=PALETTE[si % len(PALETTE)],
               edgecolor='#000', linewidth=1.1, label=name)
        for x, v in zip(xs, vals):
            ax.text(x, v, fmt.format(v), ha='center', va='bottom', fontsize=9)
    ax.set_xticks(range(len(groups)))
    ax.set_xticklabels(groups)
    _finish(ax, title, ylabel=ylabel, legend=True)


def curves(ax, series, title, xlabel, ylabel, logy=False, zero_floor=True):
    # series: [(name, xs, ys, color_or_None), ...]
    for i, (name, xs, ys, color) in enumerate(series):
        ax.plot(xs, ys, marker='o', markersize=3.5, linewidth=1.8,
                color=color or [COOL, WARM][i % 2], label=name)
    if logy:
        ax.set_yscale('log')
    elif zero_floor:
        ax.set_ylim(bottom=0)
    _finish(ax, title, xlabel=xlabel, ylabel=ylabel, legend=len(series) > 1)


print('Plot helpers ready: bars, hbars, stacked, grouped, curves, panels.')


In [ ]:
# Same cache helpers as Module 1: transformers has shipped several different cache
# representations, so normalize to a list of (key, value) tensors.
def cache_layers(past):
    if past is None:
        return []
    if hasattr(past, 'to_legacy_cache'):
        try:
            legacy = past.to_legacy_cache()
            if legacy:
                return [(lay[0], lay[1]) for lay in legacy]
        except Exception:
            pass
    if hasattr(past, 'key_cache') and hasattr(past, 'value_cache'):
        return list(zip(past.key_cache, past.value_cache))
    if hasattr(past, 'layers'):
        try:
            return [(lay.keys, lay.values) for lay in past.layers]
        except Exception:
            pass
    if isinstance(past, (list, tuple)):
        return [(lay[0], lay[1]) for lay in past if lay is not None and len(lay) >= 2]
    return []


def cache_bytes(past):
    return sum(k.numel() * k.element_size() + v.numel() * v.element_size()
               for k, v in cache_layers(past))


def cache_seq_len(past):
    if past is None:
        return 0
    if hasattr(past, 'get_seq_length'):
        try:
            return int(past.get_seq_length())
        except Exception:
            pass
    lays = cache_layers(past)
    return int(lays[0][0].shape[-2]) if lays else 0


def sync():
    if DEVICE.type == 'cuda':
        torch.cuda.synchronize()


def measure_kv_and_decode(model, n_prompt_tokens, decode_steps):
    # Prefill a synthetic prompt, weigh the resulting cache, then time decode steps.
    vocab = int(getattr(model.config, 'vocab_size', 32000))
    ids = torch.randint(0, vocab, (1, n_prompt_tokens), device=DEVICE)

    with torch.no_grad():                       # warmup
        model(input_ids=ids, use_cache=True)
    sync()

    with torch.no_grad():
        out = model(input_ids=ids, use_cache=True)
    sync()
    past = out.past_key_values
    kv_b = cache_bytes(past)
    kv_tok = max(cache_seq_len(past), 1)
    next_tok = out.logits[:, -1, :].argmax(dim=-1, keepdim=True)

    step_secs = []
    for _ in range(decode_steps):
        sync(); t0 = time.perf_counter()
        with torch.no_grad():
            out = model(input_ids=next_tok, past_key_values=past, use_cache=True)
        sync(); step_secs.append(time.perf_counter() - t0)
        past = out.past_key_values
        next_tok = out.logits[:, -1, :].argmax(dim=-1, keepdim=True)

    return {'kv_bytes': kv_b, 'kv_tokens': kv_tok, 'bytes_per_token': kv_b / kv_tok,
            'mean_itl_ms': sum(step_secs) / len(step_secs) * 1000}


print('Helpers ready.')

## 1. Why the KV Cache Is the Thing Attention Variants Attack

Multi-head attention stores one K and one V vector **per attention head, per layer, per
token**. Nothing in the maths requires that. Grouped-Query Attention (GQA) lets several
query heads share a single KV head; Multi-Query Attention (MQA) is the extreme where all
query heads share one. Multi-head Latent Attention (MLA) instead stores a single
compressed latent per token.

All three keep the query side at full width. Only the *cached* side shrinks. That is why
they cut memory and bandwidth without cutting model width.

The honest way to show this is a **controlled experiment**: identical models, one
variable changed.

## 2. Controlled Experiment: Same Model, Different Number of KV Heads

We build four small Llama-architecture models that are identical in every respect —
layers, hidden size, head dim, vocab — except for `num_key_value_heads`. Weights are
randomly initialized: we are measuring **memory layout and latency**, which depend only
on shapes, not on whether the model says anything sensible.

This isolates the one variable. Comparing two different pretrained checkpoints (§3)
cannot do that, because they differ in size too.

In [ ]:
from transformers import LlamaConfig, LlamaForCausalLM

BASE = dict(hidden_size=512, intermediate_size=1376, num_hidden_layers=8,
            num_attention_heads=8, max_position_embeddings=2048, vocab_size=32000)
HEAD_DIM = BASE['hidden_size'] // BASE['num_attention_heads']

VARIANTS = [('MHA (8 KV heads)', 8), ('GQA-4 (4 KV heads)', 4),
            ('GQA-2 (2 KV heads)', 2), ('MQA (1 KV head)', 1)]

controlled = []
for label, n_kv in VARIANTS:
    cfg = LlamaConfig(num_key_value_heads=n_kv, **BASE)
    m = LlamaForCausalLM(cfg).to(device=DEVICE, dtype=DTYPE).eval()

    r = measure_kv_and_decode(m, PROMPT_TOKENS, DECODE_STEPS)
    weight_bytes = sum(p.numel() * p.element_size() for p in m.parameters())
    controlled.append({
        'Variant': label,
        'Q heads': BASE['num_attention_heads'],
        'KV heads': n_kv,
        'Weights (MB)': round(weight_bytes / 1e6, 1),
        'KV B/token': round(r['bytes_per_token']),
        f'KV at {PROMPT_TOKENS} tok (MB)': round(r['kv_bytes'] / 1e6, 2),
        'Mean ITL (ms)': round(r['mean_itl_ms'], 2),
    })
    print(f"{label:<20} KV {r['bytes_per_token']:>8.0f} B/token | "
          f"weights {weight_bytes/1e6:7.1f} MB | ITL {r['mean_itl_ms']:6.2f} ms")

    del m
    gc.collect()
    if HAS_CUDA:
        torch.cuda.empty_cache()

controlled_table = pd.DataFrame(controlled)
mha_bpt = controlled_table['KV B/token'].iloc[0]
controlled_table['KV savings vs MHA'] = [f'{mha_bpt/b:.1f}x' for b in controlled_table['KV B/token']]
controlled_table

In [ ]:
ax1, ax2 = panels(2)
short = [v.split(' (')[0] for v in controlled_table['Variant']]

bars(ax1, short, controlled_table['KV B/token'] / 1024,
     'Measured KV cost per token\n(identical models, only KV heads differ)',
     'KB per cached token', '{:.1f} KB')
bars(ax2, short, controlled_table['Mean ITL (ms)'],
     'Measured decode latency\n(weights dominate at this tiny scale)',
     'ms per decoded token', '{:.2f}')
plt.tight_layout()
plt.show()

print('Left: KV bytes per token fall exactly in proportion to KV heads —')
for _, row in controlled_table.iterrows():
    print(f"  {row['Variant']:<20} {row['KV B/token']/1024:6.1f} KB/token  "
          f"({row['KV savings vs MHA']} vs MHA)")
print('\nRight: decode latency barely moves. Read that carefully — at 8 layers and')
print('512 hidden, the weight read dwarfs the cache read, so shrinking the cache')
print('buys almost no single-stream speed HERE. GQA pays off in CAPACITY (how many')
print('users fit) and at long context. Section 6 is where that shows up.')

## 3. Does It Hold for Real Checkpoints?

The controlled run proves the mechanism. Now confirm it on models people actually serve:
GPT-2 predates GQA and uses full MHA; TinyLlama ships GQA. Different sizes, so this is a
*confirmation*, not a controlled comparison — we normalize per layer to make it readable.

In [ ]:
REAL_MODELS = ['gpt2', 'TinyLlama/TinyLlama-1.1B-Chat-v1.0']

real_rows = []
for mid in REAL_MODELS:
    cfg = AutoConfig.from_pretrained(mid)
    n_heads = cfg.num_attention_heads
    n_kv = getattr(cfg, 'num_key_value_heads', None) or n_heads
    n_layers = cfg.num_hidden_layers
    head_dim = getattr(cfg, 'head_dim', None) or (cfg.hidden_size // n_heads)

    model = AutoModelForCausalLM.from_pretrained(mid, torch_dtype=DTYPE).to(DEVICE).eval()
    ctx = min(PROMPT_TOKENS, getattr(cfg, 'max_position_embeddings', PROMPT_TOKENS) - DECODE_STEPS - 1)
    r = measure_kv_and_decode(model, ctx, DECODE_STEPS)

    real_rows.append({
        'Model': mid.split('/')[-1],
        'Attention': 'MHA' if n_kv == n_heads else f'GQA-{n_kv}',
        'Layers': n_layers,
        'Q/KV heads': f'{n_heads}/{n_kv}',
        'Head dim': head_dim,
        'KV B/token': round(r['bytes_per_token']),
        'KV B/token/layer': round(r['bytes_per_token'] / n_layers),
        'Mean ITL (ms)': round(r['mean_itl_ms'], 2),
    })
    print(f"{mid:<40} {real_rows[-1]['Attention']:<7} "
          f"{r['bytes_per_token']:>8.0f} B/token")

    del model
    gc.collect()
    if HAS_CUDA:
        torch.cuda.empty_cache()

real_table = pd.DataFrame(real_rows)
real_table

In [ ]:
# Check the measurement against what the config predicts. If these agree, you can
# size any checkpoint's KV cache from config.json alone.
elem_bytes = torch.finfo(DTYPE).bits // 8
check = []
for mid, row in zip(REAL_MODELS, real_rows):
    cfg = AutoConfig.from_pretrained(mid)
    n_heads = cfg.num_attention_heads
    n_kv = getattr(cfg, 'num_key_value_heads', None) or n_heads
    head_dim = getattr(cfg, 'head_dim', None) or (cfg.hidden_size // n_heads)
    predicted = 2 * cfg.num_hidden_layers * n_kv * head_dim * elem_bytes
    check.append({'Model': row['Model'],
                  'Predicted B/token': predicted,
                  'Measured B/token': row['KV B/token'],
                  'Error': f"{abs(predicted - row['KV B/token'])/predicted*100:.1f}%"})

pd.DataFrame(check)

## 4. MLA: The One Number We Did Not Measure

Multi-head Latent Attention (DeepSeek-V2/V3) stores a single compressed latent per token
instead of per-head K and V. It is the biggest KV reduction in production — and it needs
DeepSeek-scale weights, which we are not downloading in a workshop.

**So the cell below is arithmetic from DeepSeek-V2's published config, not a measurement
on this machine.** It is labelled that way in the output and in the chart.

One honesty note while we are here: MLA's famous "56x" comes from comparing MLA against
*DeepSeek-V2's own* MHA configuration, which has more and larger heads than a 7B Llama.
Against a Mistral-7B-shaped MHA baseline the figure is closer to 14x. Both are true
statements about different baselines; quoting 56x next to a Mistral comparison is not.

In [ ]:
# DeepSeek-V2 published config: 512-dim compressed latent + 64-dim decoupled RoPE key,
# stored once per token per layer (not per head).
MLA_LATENT, MLA_ROPE, MLA_LAYERS = 512, 64, 60
mla_bpt_per_layer = (MLA_LATENT + MLA_ROPE) * elem_bytes

# Mistral-7B-shaped baselines, per layer, for an apples-to-apples per-layer view.
MISTRAL_HEADS, MISTRAL_KV_HEADS, MISTRAL_HEAD_DIM = 32, 8, 128
mha_ref = 2 * MISTRAL_HEADS * MISTRAL_HEAD_DIM * elem_bytes
gqa_ref = 2 * MISTRAL_KV_HEADS * MISTRAL_HEAD_DIM * elem_bytes

derived = pd.DataFrame([
    {'Mechanism': 'MHA (Mistral-shaped)', 'B/token/layer': mha_ref,
     'vs MHA': '1.0x', 'Source': 'derived from config'},
    {'Mechanism': 'GQA-8 (Mistral)', 'B/token/layer': gqa_ref,
     'vs MHA': f'{mha_ref/gqa_ref:.1f}x', 'Source': 'derived; measured in §3'},
    {'Mechanism': 'MLA (DeepSeek-V2)', 'B/token/layer': mla_bpt_per_layer,
     'vs MHA': f'{mha_ref/mla_bpt_per_layer:.1f}x', 'Source': 'derived from published config'},
])
print('--- DERIVED, NOT MEASURED (per layer, per token) ---')
display(derived)
print(f'MLA vs this MHA baseline: {mha_ref/mla_bpt_per_layer:.1f}x')
print('DeepSeek-V2 reports up to 56x against its own wider MHA baseline.')
print('\nFlashAttention is orthogonal to all of this: it cuts HBM traffic inside the')
print('attention kernel without changing how many bytes the cache occupies.')

## 5. The Other Lever: Fewer Bits per Weight

Attention variants shrink the cache. Quantization shrinks the *weights* — the larger
term in Module 1's bandwidth floor for any normal context length.

We load the same checkpoint at FP16, INT8, and INT4 and measure memory and throughput
on this GPU. Needs CUDA and `bitsandbytes`.

In [ ]:
QUANT_MODEL = 'mistralai/Mistral-7B-v0.1' if (HAS_CUDA and vram_gb >= 18) \
    else 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'
QUANT_PROMPT = 'Explain the concept of gradient descent in machine learning:'
QUANT_TOKENS = 64

quant_ready = HAS_CUDA
if quant_ready:
    try:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'bitsandbytes'])
        from transformers import BitsAndBytesConfig
        import bitsandbytes  # noqa: F401
    except Exception as exc:
        quant_ready = False
        print(f'bitsandbytes unavailable ({exc}). Section 5 will be skipped.')

print(f'Quantization target: {QUANT_MODEL}' if quant_ready
      else 'Section 5 skipped: needs a CUDA GPU with bitsandbytes.')

In [ ]:
def measure_quantized(model, tokenizer):
    # Weight memory straight from the parameter tensors: works for bnb's packed
    # int8/int4 tensors too, where element_size() reflects the stored width.
    weight_bytes = sum(p.numel() * p.element_size() for p in model.parameters())
    ids = tokenizer(QUANT_PROMPT, return_tensors='pt').to(model.device)

    with torch.no_grad():                       # warmup
        model.generate(**ids, max_new_tokens=1, pad_token_id=tokenizer.eos_token_id)
    sync(); t0 = time.perf_counter()
    with torch.no_grad():
        out = model.generate(**ids, max_new_tokens=QUANT_TOKENS, do_sample=False,
                             pad_token_id=tokenizer.eos_token_id)
    sync(); elapsed = time.perf_counter() - t0

    n_new = out.shape[1] - ids['input_ids'].shape[1]
    return {'weight_gb': weight_bytes / 1e9,
            'allocated_gb': torch.cuda.memory_allocated() / 1e9 if HAS_CUDA else 0.0,
            'throughput_tok_s': n_new / elapsed,
            'ms_per_token': elapsed / n_new * 1000}


quant_results = {}
if quant_ready:
    tokenizer_q = AutoTokenizer.from_pretrained(QUANT_MODEL)
    loaders = [
        ('FP16', dict(torch_dtype=torch.float16)),
        ('INT8', dict(quantization_config=BitsAndBytesConfig(load_in_8bit=True))),
        ('INT4', dict(quantization_config=BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_quant_type='nf4',
            bnb_4bit_compute_dtype=torch.float16))),
    ]
    for label, kwargs in loaders:
        print(f'Loading {QUANT_MODEL} in {label} ...')
        t0 = time.perf_counter()
        m = AutoModelForCausalLM.from_pretrained(QUANT_MODEL, device_map='auto', **kwargs)
        load_s = time.perf_counter() - t0

        quant_results[label] = measure_quantized(m, tokenizer_q)
        quant_results[label]['load_s'] = load_s
        r = quant_results[label]
        print(f"  weights {r['weight_gb']:5.2f} GB | {r['throughput_tok_s']:6.1f} tok/s "
              f"| {r['ms_per_token']:6.2f} ms/token | load {load_s:.1f}s")

        del m
        gc.collect()
        torch.cuda.empty_cache()

In [ ]:
if quant_results:
    fp16_gb = quant_results['FP16']['weight_gb']
    quant_table = pd.DataFrame([{
        'Precision': k,
        'Weights (GB)': round(v['weight_gb'], 2),
        'Compression': f"{fp16_gb/v['weight_gb']:.1f}x",
        'Throughput (tok/s)': round(v['throughput_tok_s'], 1),
        'ms/token': round(v['ms_per_token'], 2),
        'Load time (s)': round(v['load_s'], 1),
    } for k, v in quant_results.items()])
    display(quant_table)
    print('Smaller weights mean fewer bytes read per token — but INT8/INT4 kernels can')
    print('be slower per operation than FP16, so throughput does not always improve.')
    print('Read the ms/token column before assuming quantization is a speed win:')
    print('its reliable win is CAPACITY, and it is measured right here, not assumed.')
else:
    quant_table = None
    print('No quantization results on this machine.')

In [ ]:
if quant_results:
    ax1, ax2 = panels(2)
    precisions = list(quant_table['Precision'])

    bars(ax1, precisions, quant_table['Weights (GB)'],
         'Weights shrink with precision', 'Weight memory (GB)', '{:.2f} GB')
    bars(ax2, precisions, quant_table['ms/token'],
         'Latency does NOT automatically follow', 'ms per decoded token', '{:.2f}')
    plt.tight_layout()
    plt.show()

    for prec, comp in zip(precisions, quant_table['Compression']):
        print(f'  {prec:<6} {comp:>6} smaller weights')

## 6. Both Levers on One Axis: How Many Users Fit?

This is the question that pays for the GPU. Capacity is:

$$\text{users} \approx \frac{\text{VRAM} - \text{weights} - \text{overhead}}{\text{KV bytes per token} \times \text{context}}$$

Every term here is one we **measured**: VRAM from the device, weights from §5, KV bytes
per token from §2/§3. No spec sheets, no assumed 80 GB GPU.

In [ ]:
# Use measured inputs wherever we have them, and say so when we substitute.
if quant_results:
    weight_options = {k: v['weight_gb'] for k, v in quant_results.items()}
    weight_note = f'measured in §5 on {QUANT_MODEL}'
else:
    weight_options = {'FP16': 14.5, 'INT8': 7.2, 'INT4': 3.6}
    weight_note = 'FALLBACK estimates — §5 did not run on this machine'

if HAS_CUDA:
    budget_gb, budget_note = vram_gb, f'measured: {gpu_name}'
else:
    budget_gb, budget_note = 80.0, 'FALLBACK: assumed 80 GB (no GPU here)'

OVERHEAD_GB = 2.0
CONTEXTS = [2048, 8192, 32768]

# KV per token at a realistic 7B shape (32 layers, 32 query heads, 128 head dim),
# from the formula §3 validated against measurement. We do NOT rescale §2's toy
# numbers: §2's job was to prove the ratio holds when only KV heads change, and
# §3's was to show this formula predicts a real checkpoint's cache. Applying the
# validated formula at the real shape keeps the absolute values honest.
REF_LAYERS, REF_QHEADS, REF_HEAD_DIM = 32, 32, 128
REF_VARIANTS = [('MHA (32 KV heads)', 32), ('GQA-8 (Mistral, Llama 3)', 8),
                ('GQA-4', 4), ('MQA (1 KV head)', 1)]
kv_by_variant = {label: 2 * REF_LAYERS * n_kv * REF_HEAD_DIM * elem_bytes
                 for label, n_kv in REF_VARIANTS}

print(f'VRAM budget  : {budget_gb:.1f} GB  ({budget_note})')
print(f'Weights      : {weight_note}')
print(f'Overhead held: {OVERHEAD_GB} GB')
print(f'KV per token : formula validated in §3, applied at '
      f'{REF_LAYERS} layers / {REF_QHEADS} q-heads / {REF_HEAD_DIM} head dim')
for label, bpt in kv_by_variant.items():
    print(f'   {label:<26} {bpt/1024:7.0f} KB/token')
print()

cap_rows = []
for prec, w_gb in weight_options.items():
    for variant, bpt in kv_by_variant.items():
        for ctx in CONTEXTS:
            free = budget_gb - w_gb - OVERHEAD_GB
            per_user_gb = bpt * ctx / 1e9
            cap_rows.append({'Precision': prec, 'Attention': variant, 'Context': ctx,
                             'KV per user (GB)': round(per_user_gb, 2),
                             'Users': int(free / per_user_gb) if free > 0 else 0})

cap_table = pd.DataFrame(cap_rows)
cap_table.pivot_table(index=['Precision', 'Attention'], columns='Context',
                      values='Users', sort=False)

In [ ]:
CTX_SHOWN = 8192
variants = list(kv_by_variant)


def seats(prec, variant, ctx):
    hit = cap_table[(cap_table['Precision'] == prec) & (cap_table['Attention'] == variant)
                    & (cap_table['Context'] == ctx)]
    return int(hit['Users'].iloc[0])


ax, = panels(1, width=11, height=5)
grouped(ax, variants,
        [(f'{prec} weights', [seats(prec, v, CTX_SHOWN) for v in variants])
         for prec in weight_options],
        f'Both levers stack: attention variant x precision\n'
        f'({budget_gb:.0f} GB budget, {OVERHEAD_GB:.0f} GB held back, '
        f'{CTX_SHOWN} tokens of context)',
        'Concurrent users', '{:.0f}')
plt.tight_layout()
plt.show()

at_ctx = cap_table[cap_table['Context'] == CTX_SHOWN].sort_values('Users')
worst, best = at_ctx.iloc[0], at_ctx.iloc[-1]
print(f"At {CTX_SHOWN} tokens: {worst['Precision']} + {worst['Attention']} seats "
      f"{worst['Users']}; {best['Precision']} + {best['Attention']} seats {best['Users']} "
      f"({best['Users']/max(worst['Users'],1):.0f}x).")
print('The levers multiply because they free different bytes: quantization frees the')
print('fixed weight block once; GQA/MLA frees per-user-per-token cache forever.')

## 7. What We Measured

| Finding | Evidence |
|---------|----------|
| KV bytes per token fall exactly in proportion to KV heads | §2, controlled: identical models, only `num_key_value_heads` differs |
| Shipping checkpoints behave the same way, and `config.json` predicts their cache | §3, GPT-2 (MHA) vs TinyLlama (GQA), prediction vs measurement |
| MLA compresses further, but the headline 56x is baseline-dependent | §4 — **derived, not measured here** |
| Quantization shrinks weights 2–4x | §5, same checkpoint at FP16/INT8/INT4 |
| Lower precision does **not** guarantee lower latency | §5 ms/token column |
| The two levers stack, and capacity is where they pay off | §6, built from measured VRAM + measured weights + measured KV |

### The thing to carry out of this demo

Attention variants and quantization both buy you **capacity**, not single-stream speed.
If you need a faster single response, this is the wrong page — look at speculative
decoding and better kernels. If you need to serve more people on the GPU you already
have, this is the whole game.

### Caveats worth saying out loud

- §2 uses randomly-initialized weights. Correct for measuring layout and latency;
  it says nothing about output quality.
- §6's KV figures come from the formula, applied at a 7B shape — not from §2's toy
  models. §2 proves the ratio, §3 proves the formula matches a real checkpoint, and
  §6 applies it. Capacity is a planning estimate: it ignores fragmentation, block
  allocator granularity, and activation peaks, all of which cost you users.
- We measured no **quality** cost for INT4. It is real and task-dependent; measuring it
  needs an eval harness, which is a different demo. Do not quote INT4 capacity gains
  without it.
- All of this is unbatched pure PyTorch. A real engine does better on every absolute
  number and implements all of these levers at once.